# MASIVE-ALS — Docking en Google Colab (GPU T4 gratuita, SIN Google Drive)

**Paralelo con Kaggle:** este notebook procesa los IMPARES, Kaggle los PARES.

**Como usar:**
1. Entorno de ejecucion -> Cambiar tipo de entorno -> **T4 GPU**
2. Subir los 2 paquetes al panel de archivos (icono carpeta, izquierda):
   - `colab_receptores_plano.tar.gz`
   - `colab_ligandos50_plano.tar.gz`
   (quedan en /content)
3. Ejecutar las celdas en orden. La celda 3 los extrae sola.

**Sesion gratuita:** hasta 12 horas. Los resultados son locales a la sesion:
descargar `/content/masive_als/resultados/resultados_colab.csv` antes de cerrar.

In [ ]:
# CELDA 1: Verificar GPU
!nvidia-smi
print('GPU OK')

In [ ]:
# CELDA 2: Instalar AutoDock Vina y Open Babel
!apt-get -qq install -y openbabel > /dev/null 2>&1
!pip -q install vina meeko rdkit > /dev/null 2>&1
import vina
print('Vina OK, version:', vina.__version__)

In [ ]:
# CELDA 3: Preparar carpetas y descargar los paquetes de datos
import os, glob, tarfile, shutil, urllib.request

WORK = '/content/masive_als'
for sub in ['receptores', 'ligandos', 'resultados', 'checkpoint']:
    os.makedirs(WORK + '/' + sub, exist_ok=True)

# Los paquetes se descargan solos (repo publico de datos; si ya estan
# subidos al panel de archivos, se usan los locales)
URLS = [
    'https://raw.githubusercontent.com/fredy30-Rojas/masive-als-data/main/colab_receptores_plano.tar.gz',
    'https://raw.githubusercontent.com/fredy30-Rojas/masive-als-data/main/colab_ligandos50_plano.tar.gz',
]
nombres = [os.path.basename(t) for t in glob.glob('/content/*.tar.gz')]
for url in URLS:
    destino = '/content/' + os.path.basename(url)
    if os.path.basename(url) not in nombres:
        print('Descargando:', os.path.basename(url))
        urllib.request.urlretrieve(url, destino)
    else:
        print('Ya esta en /content:', os.path.basename(url))
tars = sorted(glob.glob('/content/*.tar.gz'))

def es_receptor(nombre):
    return nombre in ('TDP43.pdbqt', 'SOD1.pdbqt', 'FUS.pdbqt')

extract_dir = '/content/tmp_extract'
if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)
os.makedirs(extract_dir)

moved_r = moved_l = 0
for tar in tars:
    print('Extrayendo:', os.path.basename(tar))
    with tarfile.open(tar) as t:
        t.extractall(extract_dir)

# Copiar cada .pdbqt a su carpeta (aunque venga en subcarpetas)
for raiz, _, archivos in os.walk(extract_dir):
    for a in archivos:
        if a.endswith('.pdbqt'):
            destino = WORK + '/receptores/' + a if es_receptor(a) else WORK + '/ligandos/' + a
            shutil.copy(os.path.join(raiz, a), destino)
            if es_receptor(a):
                moved_r += 1
            else:
                moved_l += 1

print('Receptores: %d  |  Ligandos: %d' % (moved_r, moved_l))
print('Receptores en carpeta:', sorted(os.listdir(WORK + '/receptores')))
print('Ligandos en carpeta:', len(os.listdir(WORK + '/ligandos')))
if moved_r + moved_l == 0:
    print('ERROR: no se pudieron obtener los paquetes de datos.')

In [ ]:
# CELDA 4: Definir receptores (coordenadas de la literatura)
import os
RECEPTORES = {
    'TDP43': {
        'archivo': WORK + '/receptores/TDP43.pdbqt',
        'centro': [28.3, 43.7, 52.5],
        'tamano': [25, 25, 25]
    },
    'SOD1': {
        'archivo': WORK + '/receptores/SOD1.pdbqt',
        'centro': [27.9, 111.8, 64.4],
        'tamano': [25, 25, 25]
    },
    'FUS': {
        'archivo': WORK + '/receptores/FUS.pdbqt',
        'centro': [-14.5, 15.1, -7.8],
        'tamano': [25, 25, 25]
    }
}
for nombre, info in RECEPTORES.items():
    ok = os.path.exists(info['archivo'])
    print(nombre, 'EXISTE' if ok else 'FALTA - ejecuta la celda 3')

In [ ]:
# CELDA 5: Docking completo - todos los compuestos y receptores
import csv, glob, os, time
from vina import Vina

LIG_DIR = WORK + '/ligandos'
CSV = WORK + '/resultados/resultados_colab.csv'

# 1) Reparar FUS.pdbqt: dejar solo el primer modelo
fus = RECEPTORES['FUS']['archivo']
try:
    lines = open(fus).read().splitlines()
    keep, n_model = [], 0
    for ln in lines:
        if ln.startswith('MODEL'):
            n_model += 1
            if n_model > 1:
                break
        keep.append(ln)
    if n_model > 1:
        while keep and keep[-1].strip() == 'ENDMDL':
            keep.pop()
        keep.append('ENDMDL')
        open(fus, 'w').write('\n'.join(keep) + '\n')
        print('FUS reparado:', n_model, 'modelos -> 1')
    else:
        print('FUS sin reparar (modelos:', n_model, ')')
except Exception as ex:
    print('ERROR reparando FUS:', str(ex)[:100])

# 2) CSV inicial
if not os.path.exists(CSV):
    with open(CSV, 'w', newline='') as f:
        csv.writer(f).writerow(['ligand', 'target', 'energy', 'timestamp'])

# 3) Pares (ligand,target) pendientes
hechos = {}
if os.path.exists(CSV):
    for r in csv.DictReader(open(CSV)):
        hechos.setdefault(r['ligand'], set()).add(r['target'])

ligs = sorted(glob.glob(LIG_DIR + '/*.pdbqt'))
print('Ligandos totales:', len(ligs))
pendientes = []
for lig in ligs:
    nombre = os.path.basename(lig).replace('.pdbqt', '')
    para = hechos.get(nombre, set())
    for target in RECEPTORES:
        if target not in para:
            pendientes.append((lig, nombre, target))
print('Pares pendientes:', len(pendientes))

# 4) Acoplar los pendientes (exhaustiveness 2 para caber en la sesion)
if pendientes:
    v = Vina(sf_name='vina', verbosity=0)
    t0 = time.time()
    n_ok = 0
    for lig, nombre, target in pendientes:
        info = RECEPTORES[target]
        if not os.path.exists(info['archivo']):
            print('FALTA RECEPTOR', target)
            continue
        try:
            v.set_receptor(info['archivo'])
            v.compute_vina_maps(center=info['centro'], box_size=info['tamano'])
            v.set_ligand_from_file(lig)
            v.dock(exhaustiveness=2, n_poses=3)
            e = v.energies()[0][0]
            with open(CSV, 'a', newline='') as f:
                csv.writer(f).writerow([nombre, target, round(float(e), 4), time.strftime('%Y-%m-%d %H:%M:%S')])
            n_ok += 1
        except Exception as ex:
            print('ERROR', nombre, target, str(ex)[:80])
        if n_ok % 15 == 0:
            print('[%d acoplados] %.1f min' % (n_ok, (time.time() - t0) / 60))
    print('Acoplados ahora:', n_ok)

print()
print('=== TANDAS COMPLETADAS ===')
print('Filas en CSV:', len(list(csv.DictReader(open(CSV)))))

In [ ]:
# CELDA 6: Resumen de resultados (con respaldo completo en la celda)
import csv
rows = list(csv.DictReader(open(CSV)))
print('Total resultados:', len(rows))
if rows:
    best = sorted(rows, key=lambda x: float(x['energy']))[:10]
    print()
    print('Top 10:')
    for r in best:
        print('  ', r['ligand'], r['target'], r['energy'])

print()
print('Descargar resultados: panel de archivos -> masive_als/resultados/')
print('resultados_colab.csv -> boton derecho -> descargar')
print()
print('RESULTADOS TOTALES (respaldo en celda):')
print(open(CSV).read())